# Newton vs. FEniCSx — install everything and run the full comparison

**What this is.** One deformable soft body, simulated two ways: with **NVIDIA Newton**
(fast game/robotics solvers — XPBD, VBD, SemiImplicit, on the Warp/CUDA stack) and with
**FEniCSx / dolfinx** (gold-standard implicit **FEM**). Same mesh, same compressible
Neo-Hookean material, same gravity — *only the solver differs* — so we can measure exactly
how far Newton's fast solvers drift from an accurate FEM solve, and why. (Background: the
README and `docs/`.)

**What this notebook does.** Installs the two stacks, fetches the code, and runs the whole
pipeline scenario by scenario, showing a figure for each. As it goes, every stage appends an
**OK / ERR** line + its key numbers to **`logs/summary.txt`** — so by the end you have both
the figures *and* a one-file health report. The deeper, quantitative analysis lives in the
companion notebooks **`10_hanging_bar`**, **`20_indentation`**, **`30_convergence`**,
**`40_friction`**.

**Hardware.** The only requirement is a **CUDA GPU** (Newton runs on CUDA). The easiest
zero-setup option is **Google Colab** — free GPU, *Runtime → Change runtime type → GPU* —
but the exact same steps run on **any CUDA-capable machine** (for a local install of the FEM
side, see the README / `requirements.txt`). The meshes here are modest, so any recent GPU
works; an A100 / high-RAM runtime just adds headroom.

**How to use it.** Run the cells top to bottom: **Setup** is §1–§4, then each **scenario**
(§5–§13) runs and explains itself. Just want everything in one go? Run §1–§4, then run §5–§13.

> The Newton/Warp and FEniCSx stacks are both large and can occasionally clash in one
> runtime (numpy / PETSc / MPI). If that happens, run the Newton side and the FEM side in
> **separate notebooks** — they only ever exchange `data/*.npz` files.

## 1. Check the GPU

Newton runs on CUDA, so first confirm a GPU is visible. On Colab, set one via
*Runtime → Change runtime type → GPU* if `nvidia-smi` shows nothing.

In [ ]:
!nvidia-smi

## 2. Install Newton (and Warp)

Newton is built on **NVIDIA Warp**; installing `newton[examples]` pulls Warp in
automatically — no separate step. The last line verifies Warp initialises and sees the GPU.

In [ ]:
# NVIDIA Warp (warp-lang) is Newton's core dependency and is pulled in
# automatically by this install -- no separate Warp step is needed.
!pip install -q "newton[examples]" matplotlib
# Fallback if the name/version misbehaves (installs warp-lang explicitly):
# !pip install -q warp-lang matplotlib git+https://github.com/newton-physics/newton.git
# Verify Warp is installed and sees the GPU:
import warp as wp; wp.init(); print('warp version:', wp.config.version, '| device:', wp.get_device())

## 3. Install the FEM reference (FEniCSx / dolfinx)

The whole comparison hinges on having an **independent, accurate** solver to measure Newton
against — that is FEniCSx (implicit FEM). It is *not* a plain `pip install`: on Colab use the
**fem-on-colab** build below; on a local machine install it with
`conda install -c conda-forge fenics-dolfinx mpich pyvista` instead and skip this cell.

In [ ]:
import os

# dolfinx is NOT a plain pip install. On Colab, use the fem-on-colab build below;
# locally, prefer conda:  conda install -c conda-forge fenics-dolfinx mpich pyvista
URL = "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"
!wget -q "{URL}" -O /tmp/fenicsx.sh
assert os.path.getsize("/tmp/fenicsx.sh") > 0, "installer download failed (0 bytes) -- check URL / Internet"
!bash /tmp/fenicsx.sh
import dolfinx

print("dolfinx", dolfinx.__version__)

## 4. Get the code and install it (editable)

Fetch the repository and run `pip install -e .`. Because of the `src/` layout, the packages
(`common`, `newton_run`, `fenics_run`, `compare`) become importable **only after** this
editable install — that is what makes every `python -m …` command below work. On Colab this
clones the public repo; if you are already inside a local checkout, it just installs it.

In [ ]:
import os

REPO = 'https://github.com/AustrianTradingMachine/Newton.git'   # public repo
if os.path.exists('pyproject.toml') and os.path.isdir('src'):
    print('already inside the repo:', os.getcwd())               # local checkout -- nothing to clone
else:
    if not os.path.isdir('/content/Newton'):
        !git clone --depth 1 {REPO} /content/Newton
    %cd /content/Newton
!pip install -q -e .                                             # registers common / newton_run / fenics_run / compare
print('cwd:', os.getcwd())

### The run-helper that streams live and builds `logs/summary.txt`

Every scenario below is launched through `run(label, cmd)`, imported from
`common.runlog` (it lives in the package, not inline, so this cell is a one-liner and
the helper is testable). It **streams each stage's output live** — line by line as the
GPU run produces it, not in one chunk at the end — **and** appends a one-line **OK / ERR**
status + the key result lines to `logs/summary.txt` (full per-stage logs in
`logs/<stage>.log`). So the health report is built *as you run the scenarios*; re-running
a cell just updates that stage's entry.

In [ ]:
# The pipeline-stage runner (streams live + writes logs/summary.txt); see common/runlog.py.
# Make the repo's src/ importable in THIS already-running kernel: an editable
# `pip install -e .` writes a .pth that only a NEW process reads at startup, so the kernel
# that ran the install in section 4 won't see the packages until we add src to sys.path
# ourselves. We locate src/ robustly (cwd, the Colab clone path, or a walk-up) so it works
# regardless of the current directory.
import os
import sys


def _find_src():
    cands = [os.getcwd(), "/content/Newton"]
    p = os.getcwd()
    for _ in range(6):
        cands.append(p)
        p = os.path.dirname(p)
    for root in cands:
        src = os.path.join(root, "src")
        if os.path.isdir(os.path.join(src, "common")):
            return src
    return None


_src = _find_src()
if _src and _src not in sys.path:
    sys.path.insert(0, _src)
from common.runlog import run

print("run() ready  (src on path:", _src or "via installed package", ")")

## 5. The flagship test — the hanging bar

A soft bar is clamped at the top and **stretches under its own weight**. It is the flagship
because it is the *one* scenario with a **closed-form answer** — the 1-D self-weight bar, tip
elongation ρgL²/2E ≈ 44.3 mm — so every solver can be scored against the truth.

The headline: FEM and the implicit Newton solvers land near the analytic value; **XPBD**
settles a few percent soft, because it *projects* positions instead of solving the force
balance. We run **all three** Newton solvers here, on the *same* grid, so the verdict overlay
in §7 scores each one against FEM and the analytic bar — not XPBD alone: **XPBD** first (the
canonical run, which also writes the shared mesh `data/mesh.npz` that the FEM side reuses, so
both discretise *exactly* the same body), then **VBD** (implicit) and **SemiImplicit**
(explicit, differentiable). Each writes its own `data/newton_result*.npz` on the identical,
node-for-node mesh.

In [ ]:
# All three Newton solvers on the SAME mesh, so the §7 overlay compares every solver vs
# FEM + analytic -- not XPBD alone. Uniform solver order everywhere: XPBD -> VBD -> explicit.
# XPBD is the canonical run and writes the shared mesh data/mesh.npz; VBD (implicit) and
# SemiImplicit (explicit) then write their own npz on the identical grid.
run("Newton hanging-bar (XPBD)", "python -m newton_run.run_hanging_bar")
run("Newton hanging-bar (VBD)", "python -m newton_run.run_hanging_bar --solver vbd")
run("Newton hanging-bar (explicit)", "python -m newton_run.run_hanging_bar --solver semi_implicit")

## 6. The FEM reference (two element types)

Now the gold-standard FEM solve on the **same** mesh. `--element tet` reuses Newton's mesh
node-for-node; `--element hex` builds an independent, locking-free hex mesh as a second, more
accurate reference. Comparing both shows how much of any gap is *solver* vs. *element choice*.

In [ ]:
# tet: Newton's shared mesh (node-for-node);  hex: independent mesh (element-effect reference)
run("FEM hanging-bar tet", "python -m fenics_run.run_hanging_bar --element tet")
run("FEM hanging-bar hex", "python -m fenics_run.run_hanging_bar --element hex")

## 7. The verdict — overlay the solvers, FEM and the analytic bar

This writes the figures and a text report covering **all three** Newton solvers: tip deflection
per solver, the node-for-node error vs. FEM, and each solver's tip ratio vs. FEM (so XPBD's
softness and the implicit solvers' near-FEM agreement are read off the same table). The
displacement-profile figure overlays every solver against the analytic bar and FEM.

In [ ]:
run("Compare hanging-bar", "python -m compare.hanging_bar")
from IPython.display import Image, display

for f in ['hanging_bar_profile.png', 'hanging_bar_settling.png', 'hanging_bar_error_hist.png']:
    display(Image('figures/' + f))

## 8. Contact — a rigid sphere pressed into a soft slab

The first contact scenario, in the uniform order **Newton → FEM → overlay**. We run all
three Newton solvers on this scene (the kinematic sphere couples to the soft grid through
the shared `soft_contact` pipeline; the implicit **VBD** is the apples-to-apples match for
the implicit FEM, version-gated `TODO[verify-on-colab]`), then the FEM penalty /
Augmented-Lagrangian reference (pure UFL, anchored against analytic **Hertz**), then the
overlay.

The headline the plots make: **FEM yields a calibrated contact-force curve**
(`indentation_force.png`), while the fast positional **XPBD exposes no comparable force** —
so the honest common axis is the deformed **dimple** and the **penetration**. The FEM
variants form a coarse→accurate gradation: `tet kn×5 penalty` → `tet kn×50 penalty` →
`tet kn×50 AL` → `hex kn×50 AL` (penalty vs AL at equal kn, then tet vs locking-free hex;
AL → penetration ≈ 0).

In [ ]:
# Uniform order: Newton solvers -> FEM reference -> overlay/plots.
run("Newton indentation (XPBD)", "python -m newton_run.run_indentation")
run("Newton indentation (VBD)", "python -m newton_run.run_indentation --solver vbd")
run("Newton indentation (explicit)", "python -m newton_run.run_indentation --solver semi_implicit")
run("FEM indentation", "python -m fenics_run.run_indentation")
run("Compare indentation", "python -m compare.indentation")
import os

from IPython.display import Image, display

for f in ['indentation_force.png', 'indentation_profile.png',
          'indentation_compare_profile.png', 'indentation_newton_penetration.png']:
    if os.path.exists('figures/' + f):
        display(Image('figures/' + f))

## 9. Dynamic drop — a sphere dropped onto a block

The transient-impact scenario (gravity on), in the uniform order **Newton → FEM → overlay**:
a rigid sphere falls onto a soft block resting on the ground. All three Newton solvers run
first, then FEM **Newmark** elastodynamics + penalty contact (≈10% Kelvin–Voigt damping),
then the overlay (sphere trajectory, penetration, block strain & kinetic energy, contact
force). The implicit **VBD** is the natural partner for the implicit Newmark FEM, but the
drop's **free** sphere makes it the hardest, most version-sensitive case (two-way AVBD), and
even a clean VBD run is only a *partial* fairness fix (the transient also mixes material,
contact model and time integration). Heavier; the FEM drop's `dt` / damping are still tuned.

> The VBD/explicit drop runs need a recent Newton (`TODO[verify-on-colab]`) and may log
> **ERR** on an older pin; XPBD always records a result.

In [ ]:
# Uniform order: Newton solvers (XPBD -> VBD -> explicit) -> FEM reference -> overlay.
run("Drop Newton (XPBD)", "python -m newton_run.run_drop")
run("Drop Newton (VBD)", "python -m newton_run.run_drop --solver vbd")
run("Drop Newton (explicit)", "python -m newton_run.run_drop --solver semi_implicit")
run("Drop FEM", "python -m fenics_run.run_drop")         # FEM Newmark drop (needs dolfinx)
run("Compare drop", "python -m compare.drop")
import os

from IPython.display import Image, display

for f in ['drop_scene.png', 'drop_sphere_z.png', 'drop_penetration.png', 'drop_strain_energy.png',
          'drop_kinetic_energy.png', 'drop_contact_force.png']:
    if os.path.exists('figures/' + f):
        display(Image('figures/' + f))

## 10. The convergence study — does more budget / a finer mesh close the gap?

Uniform order **Newton → FEM → overlay**. The Newton side sweeps solver budget (XPBD
iterations / substeps) and measures the equilibrium-residual RMS; the FEM side sweeps mesh
refinement (h) and load steps. Analysis in `30_convergence`.

In [ ]:
# Uniform order: Newton -> FEM -> overlay.
run("Newton convergence", "python -m newton_run.convergence")
run("FEM convergence", "python -m fenics_run.convergence")
run("Compare convergence", "python -m compare.convergence")
import os

from IPython.display import Image, display

for f in ['convergence_newton.png', 'convergence_fem.png']:
    if os.path.exists('figures/' + f):
        display(Image('figures/' + f))

## 11. Coulomb friction — a block dragged on a floor

Uniform order **Newton → FEM → overlay**. All three Newton solvers drag the block on the
ground-contact scene (VBD/explicit version-gated `TODO[verify-on-colab]`), then the FEM run
gives the friction force + dissipated work (the force plateaus at the analytic `μ·W`), then
the overlay. Newton exposes the kinematic slip but no calibrated friction force. Analysis in
`40_friction`.

In [ ]:
# Uniform order: Newton solvers (XPBD -> VBD -> explicit) -> FEM reference -> overlay.
run("Newton friction (XPBD)", "python -m newton_run.run_friction")
run("Newton friction (VBD)", "python -m newton_run.run_friction --solver vbd")
run("Newton friction (explicit)", "python -m newton_run.run_friction --solver semi_implicit")
run("FEM friction", "python -m fenics_run.run_friction")
run("Compare friction", "python -m compare.friction")
import os

from IPython.display import Image, display

for f in ['friction_force.png', 'friction_work.png', 'friction_slip.png']:
    if os.path.exists('figures/' + f):
        display(Image('figures/' + f))

## 12. Material test — stress vs. stretch

Uniform order **Newton → FEM → overlay**. A confined uniaxial squeeze/stretch
(`F = diag(1, 1, λ)`): axial stress vs. stretch for Newton and FEM against the analytic
Neo-Hookean law, into the **large-strain** regime where the small-strain match ends.

In [ ]:
# Uniform order: Newton -> FEM -> overlay.
run("Stress-strain Newton", "python -m newton_run.run_stress_strain")
run("Stress-strain FEM", "python -m fenics_run.run_stress_strain")
run("Compare stress-strain", "python -m compare.stress_strain")
import os

from IPython.display import Image, display

if os.path.exists('figures/stress_strain.png'):
    display(Image('figures/stress_strain.png'))

## 13. Newton's distinctive trick — a differentiable stiffness fit (θ\*)

Newton is built on Warp, so the whole simulation is **differentiable**. We backpropagate
through the settling to fit an effective-stiffness multiplier **θ\*** that matches the FEM
shape — θ\* > 1 means Newton's material is effectively *softer* than FEM. (The fit runs on the
**SemiImplicit** solver, so θ\* characterises *that* solver vs. FEM, not XPBD; the learning
rate may need tuning.)

In [ ]:
run("Diffsim theta*", "python -m newton_run.diffsim")

## 14. Where to go next — the analysis notebooks

The runs above wrote all `data/*.npz`, and `logs/summary.txt` now holds the OK/ERR health
report. The **quantitative** story — deflection, energies, forces, dimples, residuals,
convergence — lives in the companion notebooks, each written as a ~10-minute, verdict-first
read:

* **`10_hanging_bar.ipynb`** — the three Newton solvers vs. FEM tet/hex vs. the analytic bar.
* **`20_indentation.ipynb`** — contact force, the dimple, penalty-vs-AL penetration.
* **`30_convergence.ipynb`** — discretisation error (FEM) vs. solver-budget error (XPBD).
* **`40_friction.ipynb`** — Coulomb friction force, dissipated work, stick → slip.